# Projection AnnDatas

For the projection of novel data onto the reference atlas only the following data is required:

- The batch column `.obs["Sample"]`
- The latent embedding `.obsm["X_embed"]`
- The per-cell cluster assignment across all hierarchical levels `.obs["L*_[ID,descr,abc_like]"]`

The full data object is 36 GB in size, of which the vast majority is taken by the count matrix `.X`. These reduced AnnDatas will therefore have no `.X` matrix and will be significantly smaller in size.

The reference neighbor adjacency `.obsp['50nn_adjacencies']` will also be pre-computed ready for projection.

A reduced object will be generated for control and all cells respectively.

In [2]:
from pathlib import Path

import anndata as ad

from dotenv import dotenv_values

In [12]:
# Load config file agnostic to project or notebook work directory
config_path = Path("config.env")
if not config_path.exists():
    config_path = Path("../../config.env")
CONFIG = dotenv_values(config_path)
assert CONFIG != {}, 'Configuration is empty - check the filepath.'

DATADIR = Path(CONFIG['DATADIR'])

# Output directory for projection anndatas
OUTDIR = DATADIR / "9_objects" / "singlecell" / "projection_anndatas"
OUTDIR.mkdir(parents=False, exist_ok=True)

In [4]:
# Load the full single-cell AnnData object
fp_full_sc_adata = DATADIR / '9_objects' / 'singlecell' / 'hypomap3d_singlecell_all.h5ad'
adata = ad.read_h5ad(fp_full_sc_adata)
adata

AnnData object with n_obs × n_vars = 1564064 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'sex_per_sample_inferred', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'L1_ID', 'L2_ID', 'L3_ID', 'L4_ID', 'L5_ID', 'L1_descr', 'L2_descr', 'L3_descr', 'L4_descr', 'L5_descr', 'L1_abc_like', 'L2_abc_like', 'L3_abc_like', 'L4_abc_like', 'L5_abc_like'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    obsm: 'X_embed', 'X_spatial_umap', 'X_umap'

In [5]:
# Remove .X count matrix
del adata.X

In [6]:
# Set aside a control-only copy of the AnnData object
adata_control = adata[adata.obs['IsControl']].copy()
adata_control

AnnData object with n_obs × n_vars = 896959 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'sex_per_sample_inferred', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'L1_ID', 'L2_ID', 'L3_ID', 'L4_ID', 'L5_ID', 'L1_descr', 'L2_descr', 'L3_descr', 'L4_descr', 'L5_descr', 'L1_abc_like', 'L2_abc_like', 'L3_abc_like', 'L4_abc_like', 'L5_abc_like'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    obsm: 'X_embed', 'X_spatial_umap', 'X_umap'

### Full AnnData Object

In [ ]:
from hm3d_utils.neighbors import knn_graph

# In order to compute bipartite SNN we need reference neighbourhoods
if '50nn_adjacencies' not in adata.obsp.keys():

    D = knn_graph(
        adata.obsm['X_embed'],
        k=50+1, # +1 for self reference
        use_gpu=True
    )

    adata.obsp['50nn_adjacencies'] = (D > 0) # convert to boolean and drop self (d=0)

adata.obsp['50nn_adjacencies']

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 78203200 stored elements and shape (1564064, 1564064)>

In [13]:
outfile = OUTDIR / "hypomap3d_sc_all_projection_only.h5ad"
adata.write(outfile)

### Control-only AnnData Object

In [ ]:
# In order to compute bipartite SNN we need reference neighbourhoods
if '50nn_adjacencies' not in adata_control.obsp.keys():

    D = knn_graph(
        adata_control.obsm['X_embed'],
        k=50+1, # +1 for self reference
        use_gpu=True
    )

    adata_control.obsp['50nn_adjacencies'] = (D > 0) # convert to boolean and drop self (d=0)

adata_control.obsp['50nn_adjacencies']

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 44847950 stored elements and shape (896959, 896959)>

In [14]:
outfile = OUTDIR / "hypomap3d_sc_control_projection_only.h5ad"
adata_control.write(outfile)